# 신경망 프루닝(가지치기) 실습 — Optimal Brain Damage에서 2:4 Sparsity까지

파이토치로 MNIST를 MLP로 학습한 뒤, **덜 중요한 가중치를 잘라내(pruning)** 모델을 얼마나 줄일 수 있는지 직접 측정합니다.

1. 무엇을 자를까 — **magnitude**(|w|) vs **OBD saliency**(2차/헤시안) vs random
2. 프루닝 + **재학습**으로 정확도를 얼마나 되살리나 (놀라운 압축률)
3. **2:4 structured sparsity** — NVIDIA Ampere Sparse Tensor Core가 실제로 가속하는 패턴
4. dense vs 2:4 sparse 행렬곱 **속도 벤치마크** (하드웨어 지원의 조건)

> 참고 논문: LeCun, Denker, Solla, *Optimal Brain Damage*, NeurIPS 1989 · Han et al., *Deep Compression*, ICLR 2016 · Mishra et al., *Accelerating Sparse DNNs*, 2021.
> 2:4 속도 가속은 **compute capability ≥ 8.0 (Ampere, 예: A100)** 이 필요합니다. Colab에서 GPU 런타임(가능하면 A100)을 선택하세요.

## 0. 준비

In [ ]:
import copy, time
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
from torchvision import transforms
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cap = torch.cuda.get_device_capability() if device.type == "cuda" else (0, 0)
gpu = torch.cuda.get_device_name() if device.type == "cuda" else "CPU"
print("device:", device, "|", gpu, "| capability:", cap)

## 1. MNIST를 MLP로 학습

작은 3층 MLP(784 → 512 → 512 → 10)를 몇 에폭 학습합니다. 목적은 최고 성능이 아니라 **프루닝해 볼 가중치**를 얻는 것입니다.

In [ ]:
transform = transforms.Compose([transforms.ToTensor(),
                                transforms.Normalize((0.1307,), (0.3081,))])
train_ds = torchvision.datasets.MNIST("./data", train=True,  download=True, transform=transform)
test_ds  = torchvision.datasets.MNIST("./data", train=False, download=True, transform=transform)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False)

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28*28, 512)
        self.fc2 = nn.Linear(512, 512)
        self.fc3 = nn.Linear(512, 10)
    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x)); x = F.relu(self.fc2(x)); return self.fc3(x)

def accuracy(m):
    m.eval(); c = t = 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            c += (m(x).argmax(1) == y).sum().item(); t += y.size(0)
    return c / t

def train_epochs(m, epochs, lr=1e-3, masks=None):
    opt = torch.optim.Adam(m.parameters(), lr=lr)
    for _ in range(epochs):
        m.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(); F.cross_entropy(m(x), y).backward(); opt.step()
            if masks:                       # 재학습 중 잘린 가중치는 계속 0으로 유지
                with torch.no_grad():
                    for name, p in m.named_parameters():
                        if name in masks: p.mul_(masks[name])

model = MLP().to(device)
train_epochs(model, 3)
base_acc = accuracy(model)
print("baseline (dense) accuracy:", base_acc)

PRUNABLE = ["fc1.weight", "fc2.weight", "fc3.weight"]   # 편향(bias)은 두고 가중치만 프루닝

## 2. 무엇을 자를까 — 중요도(importance) 기준

프루닝의 첫 질문은 "어떤 가중치를 지울 것인가"입니다. 세 가지 기준을 비교합니다.

- **Magnitude**: 중요도 = |w|. 작으면 지운다. 싸고 강력해서 사실상 표준.
- **OBD saliency** (LeCun 1989): 각 가중치를 지웠을 때 손실 증가량을 2차 미분으로 근사 → `saliency = ½·H_ii·w²`. 여기서 대각 헤시안 `H_ii`는 **empirical Fisher**(기울기 제곱의 평균)로 근사합니다(헤시안 전체는 비싸므로).
- **Random**: 무작위. 비교용 하한선.

전역(global) 프루닝: 세 층의 가중치를 한데 모아 중요도 하위 `sparsity` 비율을 0으로 만듭니다.

In [ ]:
def magnitude_scores(m):
    return {n: p.detach().abs() for n, p in m.named_parameters() if n in PRUNABLE}

def obd_saliency(m, n_batches=40):
    """OBD saliency s_i = ½·H_ii·w_i², 대각 헤시안 H_ii ≈ empirical Fisher(기울기²의 평균)."""
    fisher = {n: torch.zeros_like(p) for n, p in m.named_parameters() if n in PRUNABLE}
    m.eval(); nb = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        m.zero_grad(); F.cross_entropy(m(x), y).backward()
        for n, p in m.named_parameters():
            if n in PRUNABLE: fisher[n] += p.grad.detach() ** 2
        nb += 1
        if nb >= n_batches: break
    return {n: 0.5 * (fisher[n] / nb) * (p.detach() ** 2)
            for n, p in m.named_parameters() if n in PRUNABLE}

def rand_scores(m):
    return {n: torch.rand_like(p) for n, p in m.named_parameters() if n in PRUNABLE}

def global_mask(scores, sparsity):
    """중요도 하위 sparsity 비율을 자르는 0/1 마스크(전역 임계값)."""
    alls = torch.cat([s.flatten() for s in scores.values()])
    k = int(sparsity * alls.numel())
    if k == 0: return {n: torch.ones_like(s) for n, s in scores.items()}
    thr = torch.kthvalue(alls, k).values
    return {n: (s > thr).float() for n, s in scores.items()}

def apply_masks(m, masks):
    with torch.no_grad():
        for n, p in m.named_parameters():
            if n in masks: p.mul_(masks[n])

def eval_pruned(scores_fn, sparsity, retrain_epochs=0):
    m = copy.deepcopy(model)
    masks = global_mask(scores_fn(m), sparsity)
    apply_masks(m, masks)
    if retrain_epochs: train_epochs(m, retrain_epochs, lr=1e-4, masks=masks)
    return accuracy(m)

In [ ]:
sparsities = [0.0, 0.3, 0.5, 0.7, 0.8, 0.9, 0.95, 0.98]
acc_obd  = [eval_pruned(obd_saliency, s)    for s in sparsities]
acc_mag  = [eval_pruned(magnitude_scores, s) for s in sparsities]
acc_rand = [eval_pruned(rand_scores, s)     for s in sparsities]
for s, a1, a2, a3 in zip(sparsities, acc_obd, acc_mag, acc_rand):
    print(f"sparsity {s:4.0%} | OBD {a1:.4f} | magnitude {a2:.4f} | random {a3:.4f}")

xs = [s*100 for s in sparsities]
plt.figure(figsize=(6.8, 4.3))
plt.axhline(base_acc, color="gray", ls=":", label=f"dense baseline ({base_acc:.3f})")
plt.plot(xs, acc_obd,  "o-",  color="#ef4444", label="OBD saliency (½·H·w²)")
plt.plot(xs, acc_mag,  "s-",  color="#3b82f6", label="magnitude (|w|)")
plt.plot(xs, acc_rand, "^--", color="#9ca3af", label="random")
plt.title("One-shot pruning (no retrain): what to cut matters")
plt.xlabel("sparsity (% of weights removed)"); plt.ylabel("test accuracy")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

> 관찰: **magnitude와 OBD는 거의 겹치고, random은 50%만 넘어도 무너집니다.** 즉 "무엇을 자르는지"가 결정적입니다. OBD가 도입한 2차(헤시안) 통찰은 극단적 sparsity에서 빛나지만, magnitude가 워낙 싸고 강력해서 실무 표준이 된 이유도 여기서 보입니다.

## 3. 프루닝 + 재학습 — 놀라운 압축률

한 번에 많이 자르면 정확도가 떨어지지만, **자르고 → 재학습**을 반복하면 놀라울 만큼 높은 sparsity까지 정확도가 유지됩니다. (재학습 학습률은 보통 원래의 1/10~1/100)

- **prune only**: 자르고 끝 (재학습 X)
- **prune once + retrain**: 목표 sparsity로 한 번에 자른 뒤 재학습
- **iterative prune + retrain**: 조금씩 sparsity를 올리며 매번 재학습 (모델을 이어감)

In [ ]:
iter_sparsities = [0.3, 0.5, 0.7, 0.8, 0.9, 0.95, 0.98]
acc_noretrain   = [acc_mag[sparsities.index(s)] for s in iter_sparsities]
acc_oneshot_rt  = [eval_pruned(magnitude_scores, s, retrain_epochs=1) for s in iter_sparsities]

m_it = copy.deepcopy(model); acc_iter = []
for s in iter_sparsities:
    masks = global_mask(magnitude_scores(m_it), s)
    apply_masks(m_it, masks)
    train_epochs(m_it, 1, lr=1e-4, masks=masks)
    acc_iter.append(accuracy(m_it))
    print(f"iterative sparsity {s:4.0%} -> acc {acc_iter[-1]:.4f}")

xs = [s*100 for s in iter_sparsities]
plt.figure(figsize=(6.8, 4.3))
plt.axhline(base_acc, color="gray", ls=":", label=f"dense baseline ({base_acc:.3f})")
plt.plot(xs, acc_noretrain,  "^--", color="#9ca3af", label="prune only (no retrain)")
plt.plot(xs, acc_oneshot_rt, "s-",  color="#e0a020", label="prune once + retrain")
plt.plot(xs, acc_iter,       "o-",  color="#22c55e", label="iterative prune + retrain")
plt.title("Retraining recovers accuracy at surprisingly high sparsity")
plt.xlabel("sparsity (% of weights removed)"); plt.ylabel("test accuracy")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

> 관찰: iterative prune+retrain은 **가중치를 95% 잘라내도(20배 적은 파라미터)** dense와 거의 같은 정확도를 유지합니다. Deep Compression(Han 2016)이 AlexNet을 9~35배 줄인 것과 같은 원리입니다.

## 4. 2:4 Structured Sparsity — 하드웨어가 가속하는 패턴

위의 unstructured 프루닝은 압축률은 좋지만, 0의 위치가 불규칙해서 **GPU가 실제로 가속하기 어렵습니다.** NVIDIA Ampere+ 는 **2:4 패턴**(연속된 4개 중 정확히 2개가 0)만 Sparse Tensor Core로 가속합니다.

magnitude로 4열마다 작은 2개를 자르는 2:4 마스크를 만들고, 정확도를 봅니다.

In [ ]:
def two_four_mask(weight):
    """4열 단위로 |w|가 큰 2개만 남기는 2:4 마스크."""
    w = weight.detach().abs(); R, C = w.shape
    mask = torch.zeros_like(weight)
    for j in range(0, C - C % 4, 4):
        idx = w[:, j:j+4].topk(2, dim=1).indices
        mask[:, j:j+4].scatter_(1, idx, 1.0)
    if C % 4: mask[:, C - C % 4:] = 1.0
    return mask

def eval_2to4(retrain_epochs=0):
    m = copy.deepcopy(model); masks = {}
    with torch.no_grad():
        for n, p in m.named_parameters():
            if n in ["fc1.weight", "fc2.weight"]:      # 512열 → 4의 배수
                mk = two_four_mask(p); masks[n] = mk; p.mul_(mk)
    if retrain_epochs: train_epochs(m, retrain_epochs, lr=1e-4, masks=masks)
    return accuracy(m)

print(f"dense           : {base_acc:.4f}")
print(f"2:4 (no retrain): {eval_2to4(0):.4f}")
print(f"2:4 (+retrain)  : {eval_2to4(1):.4f}")

### 파이썬에서 2:4 sparse 행렬곱 — `to_sparse_semi_structured`

PyTorch는 2:4 패턴을 압축 저장하고 Sparse Tensor Core로 곱하는 API를 제공합니다. (fp16, Ampere+ 필요)

In [ ]:
from torch.sparse import to_sparse_semi_structured

# 2:4 패턴을 만족하는 예시 행렬 (각 4개 중 2개가 0)
mask = torch.Tensor([0, 0, 1, 1]).tile((64, 16)).bool()
W = (torch.rand(64, 64) * mask).half().cuda()
Ws = to_sparse_semi_structured(W)          # 비영값 + 2-bit 인덱스로 압축
x  = torch.rand(64, 64).half().cuda()

print("dense  == sparse 결과 일치:", torch.allclose(torch.mm(W, x), torch.mm(Ws, x), atol=1e-2))
print("압축 표현:", type(Ws).__name__)

### dense vs 2:4 sparse 행렬곱 속도

2:4의 가속은 **큰 행렬 + Ampere+ 하드웨어**에서만 나타납니다. 작은 행렬은 오히려 오버헤드로 느립니다 — "프루닝은 하드웨어와 워크로드가 받쳐줘야 이득"이라는 걸 직접 확인합니다.

In [ ]:
if device.type == "cuda" and cap[0] >= 8:
    sizes, speedups = [1024, 2048, 4096, 8192, 16384], []
    for N in sizes:
        m4 = torch.zeros(N, N); m4[:, ::4] = 1; m4[:, 1::4] = 1
        A = (torch.rand(N, N) * m4).half().cuda()
        B = torch.rand(N, N).half().cuda()
        As = to_sparse_semi_structured(A)
        for _ in range(3): torch.mm(A, B); torch.mm(As, B)      # warmup
        torch.cuda.synchronize(); t0 = time.time()
        for _ in range(20): torch.mm(A, B)
        torch.cuda.synchronize(); dt_d = (time.time()-t0)/20*1e3
        t0 = time.time()
        for _ in range(20): torch.mm(As, B)
        torch.cuda.synchronize(); dt_s = (time.time()-t0)/20*1e3
        speedups.append(dt_d/dt_s)
        print(f"N={N:5d}  dense={dt_d:.3f}ms  2:4={dt_s:.3f}ms  speedup={dt_d/dt_s:.2f}x")
    plt.figure(figsize=(6.8, 4.3))
    plt.plot(sizes, speedups, "o-", color="#22c55e")
    plt.axhline(1.0, color="gray", ls=":", label="dense (1×)")
    plt.title(f"2:4 sparse vs dense fp16 matmul on {gpu}")
    plt.xlabel("matrix size N"); plt.ylabel("speedup over dense")
    plt.xscale("log", base=2); plt.legend(); plt.grid(alpha=0.3, which="both")
    plt.tight_layout(); plt.show()
else:
    print(f"2:4 가속은 compute capability >= 8.0 (Ampere) 필요. 현재 {gpu} cap {cap}.")

## 마무리 & 더 해보기

- **무엇을 자를까**: magnitude(|w|)가 싸고 강력. OBD의 2차 saliency는 극단적 sparsity에서 이점. 둘 다 random을 압도.
- **재학습**: prune→retrain 반복이면 95% sparsity(20배)에서도 정확도 유지.
- **2:4 structured**: unstructured와 달리 Ampere Sparse Tensor Core가 실제 가속. 단, 큰 행렬에서만 이득.

**더 해볼 것**
1. `torch.ao.pruning`(SparsityConfig)으로 프루닝 파이프라인 자동화
2. Conv 층이 있는 모델(LeNet/VGG)로 확장, 채널 프루닝
3. sensitivity analysis로 층별 프루닝 비율 자동 결정 (AMC, NetAdapt)
4. 프루닝 + 양자화 결합 (Deep Compression 전체 파이프라인)